# AI Model Validation: From Mathematics to Trust
## Episode 06 — Cross-Validation, Resampling, and How Much Evidence Is Enough?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

A model score is not a physical constant. Change the evaluation sample and the score can change.

> **How much evidence do we need before believing a measured model-performance number?**

This episode develops holdout evaluation, K-fold and repeated cross-validation, bootstrap resampling, confidence intervals, paired model comparison, nested cross-validation, rare-failure evidence, and the limits of resampling.


## 1. A Metric Is an Estimate

For loss L, finite-sample test risk is:

### R̂test(f) = (1/n) Σᵢ₌₁ⁿ L(𝑦ᵢ, f(𝑥ᵢ))

Population risk is:

### R(f) = E[L(Y, f(X))]

Therefore:

### R̂test(f) ≠ R(f)

A finite-sample metric is evidence about population performance, not population performance itself.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys
print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 2. Same Model, Different Test Samples


In [ ]:
rng=np.random.default_rng(42)
def truth(x): return 2*x+1
def model(x): return 1.9*x+1.2
def mse(y,p): return np.mean((y-p)**2)

scores=[]
for _ in range(1000):
    x=rng.normal(0,1,50)
    y=truth(x)+rng.normal(0,1,50)
    scores.append(mse(y,model(x)))
scores=np.array(scores)

print("Mean MSE:",scores.mean())
print("SD MSE  :",scores.std(ddof=1))
print("Range   :",scores.min(),scores.max())

plt.figure(figsize=(8,5))
plt.hist(scores,bins=40)
plt.xlabel("Observed test MSE"); plt.ylabel("Frequency")
plt.title("Same Model, Different Evaluation Samples")
plt.grid(alpha=.2); plt.show()


The model never changed. The score changed because the evaluation sample changed. This is **sampling variability**.


## 3. Standard Error

For an estimator T̂:

### SE(T̂) = √Var(T̂)

For an independent sample mean:

### SE(X̄) = σ / √n

With unknown σ:

### ŜE(X̄) = s / √n

The inverse square-root relationship means that substantially reducing uncertainty can require substantially more data.


## 4. Sample Size and Metric Stability


In [ ]:
rng=np.random.default_rng(7)
sizes=[20,50,100,500,2000]; spread=[]
for n in sizes:
    est=[]
    for _ in range(1000):
        x=rng.normal(0,1,n)
        y=truth(x)+rng.normal(0,1,n)
        est.append(mse(y,model(x)))
    spread.append(np.std(est,ddof=1))
    print(n,spread[-1])

plt.figure(figsize=(8,5))
plt.plot(sizes,spread,marker="o")
plt.xlabel("Evaluation sample size")
plt.ylabel("SD of observed MSE")
plt.title("Sample Size and Stability")
plt.grid(alpha=.25); plt.show()


More data generally reduces sampling variability. It does not repair biased sampling, leakage, distribution shift, or poor measurement design.


## 5. Holdout Validation

A simple evaluation separates training and testing data:

### D = Dtrain ∪ Dtest

### Dtrain ∩ Dtest = ∅

The test set should not influence model fitting. One holdout is efficient, but its result can depend strongly on the particular split.


## 6. K-Fold Cross-Validation

K-fold CV divides data into K non-overlapping folds. Each fold is used once for validation while the remaining folds are used for training.

### CVₖ = (1/K) Σₖ₌₁ᴷ Lₖ

Every observation participates in validation once and training K−1 times.


In [ ]:
rng=np.random.default_rng(123)
n=180
x=rng.uniform(-3,3,n)
y=.5*x**3-x+rng.normal(0,.8,n)

def poly(xtr,ytr,d): return np.poly1d(np.polyfit(xtr,ytr,d))
def folds(n,k,seed):
    r=np.random.default_rng(seed)
    return np.array_split(r.permutation(n),k)

def cv(x,y,d,k=5,seed=1):
    fs=folds(len(x),k,seed); losses=[]
    for i in range(k):
        val=fs[i]
        train=np.concatenate([fs[j] for j in range(k) if j!=i])
        f=poly(x[train],y[train],d)
        losses.append(mse(y[val],f(x[val])))
    return np.array(losses)

s=cv(x,y,3,5,42)
print("Fold MSE:",np.round(s,4))
print("Mean:",s.mean())
print("Fold SD:",s.std(ddof=1))


## 7. What Does K Control?

The training fraction in each K-fold iteration is approximately:

### (K − 1) / K

K=5 uses about 80% for training per fold; K=10 uses about 90%.

There is no universally optimal K. The choice depends on data size, computational cost, dependence structure, and validation objective.


In [ ]:
for k in [3,5,10,20]:
    s=cv(x,y,3,k,42)
    print(f"K={k:>2} mean={s.mean():.4f} fold-SD={s.std(ddof=1):.4f}")


## 8. Fold SD Is Not Automatically a Confidence Interval

A tempting quantity is:

### sfold / √K

But K-fold validation losses are not generally independent because their training sets overlap.

Naively treating folds as independent experimental replicates can therefore produce misleading uncertainty estimates.


## 9. Repeated K-Fold Cross-Validation

Repeat K-fold CV using different partitions. If repetition r produces Cᵣ:

### C̄ = (1/R) Σᵣ₌₁ᴿ Cᵣ

The resulting distribution exposes sensitivity to the partitioning procedure.


In [ ]:
rep=[]
for seed in range(100):
    rep.append(cv(x,y,3,5,seed).mean())
rep=np.array(rep)
print("Mean:",rep.mean())
print("SD:",rep.std(ddof=1))
print("2.5%, 97.5%:",np.quantile(rep,[.025,.975]))

plt.figure(figsize=(8,5))
plt.hist(rep,bins=25)
plt.xlabel("Mean 5-fold CV MSE"); plt.ylabel("Frequency")
plt.title("Partition Sensitivity")
plt.grid(alpha=.2); plt.show()


Repeated CV exposes split sensitivity. It does not create new independent populations or guarantee deployment performance.


## 10. Leave-One-Out Cross-Validation

LOOCV is the limiting K-fold case:

### K = n

Each model is trained on n−1 observations and tested on the remaining observation.

LOOCV uses almost all data for each fit, but can be computationally expensive and is not automatically superior to ordinary K-fold CV.


## 11. Bootstrap Resampling

A bootstrap sample contains n draws **with replacement** from the observed n cases.

For replicate b:

### θ̂*(b) = T(D*(b))

Repeating this B times approximates the sampling behaviour of the statistic conditional on the observed data.


## 12. Why 63.2%?

For one observation:

### P(not selected) = (1 − 1/n)ⁿ

As n grows:

### (1 − 1/n)ⁿ → e⁻¹ ≈ 0.368

So the expected fraction represented at least once approaches:

### 1 − e⁻¹ ≈ 0.632


In [ ]:
rng=np.random.default_rng(99)
for n in [20,100,1000,10000]:
    u=[]
    for _ in range(1000):
        sample=rng.integers(0,n,n)
        u.append(len(np.unique(sample))/n)
    print(f"n={n:>5} mean unique fraction={np.mean(u):.4f}")


## 13. Bootstrap Uncertainty for Test MSE


In [ ]:
rng=np.random.default_rng(2026)
xt=rng.normal(0,1,120)
yt=truth(xt)+rng.normal(0,1,120)
loss=(yt-model(xt))**2
observed=loss.mean()

B=5000
boot=np.empty(B)
for b in range(B):
    boot[b]=rng.choice(loss,size=len(loss),replace=True).mean()

lo,hi=np.quantile(boot,[.025,.975])
print(f"Observed MSE={observed:.4f}")
print(f"Bootstrap 95% percentile interval=[{lo:.4f},{hi:.4f}]")

plt.figure(figsize=(8,5))
plt.hist(boot,bins=50)
plt.axvline(observed,linestyle="--",label="Observed")
plt.axvline(lo,linestyle=":"); plt.axvline(hi,linestyle=":")
plt.xlabel("Bootstrap MSE"); plt.ylabel("Frequency")
plt.title("Bootstrap Distribution")
plt.legend(); plt.grid(alpha=.2); plt.show()


The percentile interval is conditional on the observed sample and bootstrap assumptions. It is not a guarantee of future deployment performance.


## 14. Confidence Interval Interpretation

A frequentist 95% confidence interval describes a repeated procedure: under its assumptions, approximately 95% of intervals constructed over repeated samples would contain the target parameter.

It does not generally mean there is a 95% probability that an already-computed fixed interval contains a fixed parameter.


## 15. Paired Model Comparison

If models A and B are evaluated on the same cases, define:

### dᵢ = Lᵢ(A) − Lᵢ(B)

and:

### d̄ = (1/n) Σᵢ₌₁ⁿ dᵢ

Paired differences preserve the fact that both models faced the same observations.


In [ ]:
rng=np.random.default_rng(17)
n=150
x2=rng.normal(0,1,n)
y2=truth(x2)+rng.normal(0,1,n)
pa=1.8*x2+1.1
pb=2.05*x2+.9
la=(y2-pa)**2
lb=(y2-pb)**2
d=la-lb
print("A MSE:",la.mean())
print("B MSE:",lb.mean())
print("Mean paired difference A-B:",d.mean())


## 16. Uncertainty of a Paired Difference

For independent evaluation cases:

### ŜE(d̄) = sd / √n

An approximate large-sample interval is:

### d̄ ± z₁₋α/₂ (sd / √n)

For a nominal 95% normal approximation, z ≈ 1.96.

Clustered, repeated, grouped, or time-dependent observations require methods that preserve those dependencies.


In [ ]:
mean_d=d.mean()
se=d.std(ddof=1)/np.sqrt(n)
print("Mean difference:",mean_d)
print("Approx. 95% interval:",(mean_d-1.96*se,mean_d+1.96*se))


## 17. Paired Bootstrap Model Comparison


In [ ]:
rng=np.random.default_rng(18)
B=5000
bd=np.empty(B)
for b in range(B):
    idx=rng.integers(0,n,n)
    bd[b]=np.mean(la[idx]-lb[idx])
lo,hi=np.quantile(bd,[.025,.975])
print("Paired bootstrap interval:",(lo,hi))

plt.figure(figsize=(8,5))
plt.hist(bd,bins=50)
plt.axvline(0,linestyle="--",label="No difference")
plt.xlabel("Mean loss difference A-B"); plt.ylabel("Frequency")
plt.title("Uncertainty in Model Comparison")
plt.legend(); plt.grid(alpha=.2); plt.show()


## 18. Statistical Evidence ≠ Practical Importance

A tiny improvement can become statistically detectable with a sufficiently large sample.

### Statistical Evidence ≠ Practical Significance

Model validation should therefore report effect magnitude, uncertainty, and decision consequences—not merely whether a statistical threshold was crossed.


## 19. Multiple Model Selection and the Winner's Curse

If many candidates are tested and the one with the minimum observed validation loss is selected:

### minⱼ R̂validation(fⱼ)

random variation helps determine the winner.

The selected validation score can therefore be optimistic.


In [ ]:
rng=np.random.default_rng(1234)
true_loss=1.0
for count in [1,5,20,100,500]:
    winners=[]
    for _ in range(5000):
        observed=true_loss+rng.normal(0,.15,count)
        winners.append(observed.min())
    print(f"Candidates={count:>3} mean best observed loss={np.mean(winners):.4f}")


All candidates in this simulation have identical true expected loss. Trying more candidates still produces an increasingly impressive apparent winner.


## 20. Nested Cross-Validation

Nested CV separates model selection from evaluation:

### Outer Evaluation [ Inner Model Selection ]

The inner loop chooses hyperparameters or model class.

The outer loop evaluates the entire selection procedure on data untouched by that inner selection.


In [ ]:
rng=np.random.default_rng(314)
xn=rng.uniform(-3,3,180)
yn=.5*xn**3-xn+rng.normal(0,.8,180)

outer=folds(len(xn),5,10)
degrees=[1,2,3,4,5,8]
outer_loss=[]; chosen=[]

for i in range(5):
    test=outer[i]
    train=np.concatenate([outer[j] for j in range(5) if j!=i])
    inner_scores=[]
    for degree in degrees:
        inner_scores.append(cv(xn[train],yn[train],degree,4,100+i).mean())
    best=degrees[int(np.argmin(inner_scores))]
    chosen.append(best)
    f=poly(xn[train],yn[train],best)
    outer_loss.append(mse(yn[test],f(xn[test])))

print("Chosen degrees:",chosen)
print("Outer losses:",np.round(outer_loss,4))
print("Nested CV estimate:",np.mean(outer_loss))


## 21. Splitting Must Match the Deployment Question

Random K-fold CV can be invalid for:

- time series
- repeated measurements
- multiple records from the same customer
- multiple images from one patient
- documents from one source
- geographic clusters

If deployment means predicting for unseen groups:

### Gtrain ∩ Gtest = ∅

For forecasting, chronology should be respected:

### ttrain < tvalidation < ttest

The evaluation protocol should imitate the information constraints of real deployment.


## 22. Resampling Cannot Repair Bad Data

Cross-validation and bootstrap reuse the observed dataset.

### Resampling More ≠ Collecting Better Evidence

If the original data are unrepresentative, contaminated, biased, or missing critical populations, computational resampling does not manufacture the missing information.


## 23. How Much Evidence Is Enough?

There is no universal number of folds or samples that makes a model “validated.”

Evidence requirements depend on expected effect size, acceptable uncertainty, rare-failure prevalence, consequences of errors, population heterogeneity, subgroup requirements, deployment environment, and safety or regulatory requirements.


## 24. Rare Failures

If a harmful failure has probability p, the probability of seeing zero failures in n independent trials is:

### P(X = 0) = (1 − p)ⁿ

No observed failures therefore do not prove zero failure probability.


In [ ]:
rates=[.10,.01,.001,.0001]
sizes=[100,1000,10000]
for p in rates:
    print(f"\nTrue failure probability={p:.4%}")
    for n in sizes:
        print(f"n={n:>5}: P(zero observed)={(1-p)**n:.6f}")


## 25. Rule of Three

When zero failures are observed in n independent Bernoulli trials, a useful approximate 95% upper bound for a small failure probability is:

### pupper ≈ 3/n

For 1,000 observations with zero failures, this approximation gives 0.3%, not 0%.


In [ ]:
for n in [100,1000,10000,100000]:
    print(f"n={n:>6} -> approximate 95% upper failure rate={3/n:.4%}")


## 26. Evidence Is Claim-Dependent

“How many samples do I need?” is incomplete.

A stronger scientific question is:

> How much data do I need to estimate the quantity I care about with uncertainty small enough for the intended decision?

The validation claim should determine the evidence requirement.


## 27. Internal vs External Validation

Cross-validation partitions the same underlying dataset.

External validation evaluates independently collected data relevant to intended deployment.

### Internal Resampling ≠ External Validation

Both are valuable, but they answer different questions.


## 28. Validation Has a Time Dimension

A model validated at one time can later encounter population drift, concept drift, new failure modes, adversarial behaviour, altered data quality, or changed dependencies.

### Validation(t₀) ⇏ Validation(t₁)

Pre-deployment validation therefore cannot replace post-deployment monitoring.


## 29. OpenVals Perspective

For every important metric, an AI validation platform should preserve:

- evaluation dataset and sample size
- split strategy
- fold-level results
- repeated-run variability
- uncertainty intervals
- paired model differences
- model-selection procedure
- leakage controls
- group and temporal constraints
- rare-event coverage
- external-validation status
- deployment-monitoring status

The goal is not more numbers. It is the **strength of evidence behind each claim**.


## 30. Scientific Evidence Chain

### Metric → Sampling Variability → Resampling → Uncertainty → Comparison → Evidence

A performance number without uncertainty can create false precision.

A model comparison without paired evidence can create false confidence.

A validation procedure that ignores selection can create optimistic conclusions.


## 31. Mathematical Summary

### R̂test(f) = (1/n) Σᵢ₌₁ⁿ L(𝑦ᵢ, f(𝑥ᵢ))

### R(f) = E[L(Y, f(X))]

### CVₖ = (1/K) Σₖ₌₁ᴷ Lₖ

### (1 − 1/n)ⁿ → e⁻¹ ≈ 0.368

### 1 − e⁻¹ ≈ 0.632

### dᵢ = Lᵢ(A) − Lᵢ(B)

### ŜE(d̄) = sd / √n

### P(X = 0) = (1 − p)ⁿ

### pupper ≈ 3/n


## 32. Central Conclusion

### Performance Number ≠ Strength of Evidence

> **A model metric tells us what happened in an evaluation. Statistical validation tells us how much confidence that evaluation deserves.**

Cross-validation does not prove generalization.

Bootstrap intervals do not guarantee future performance.

More folds do not repair poor sampling.

Zero observed failures do not prove zero risk.

The scientific objective is to quantify what we know, how uncertain we are, and whether the evidence is strong enough for the decision being made.

### Next Episode

**Episode 07 — Classification Metrics: Why Accuracy Can Be Completely Misleading**

The next episode will build the confusion matrix from first principles and derive accuracy, precision, recall, specificity, F1 score, false-positive rate, false-negative rate, prevalence, balanced accuracy, and threshold-dependent behaviour.
